In [2]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split

DATASET_PATH = "/kaggle/input/datasets/saitejaswinichikoti/plantvillage-subset"

SUBSET_FOLDER_PARSER = {
    # Apple (0)
    "Apple___Apple_scab": (0, 5),
    "Apple___Black_rot": (0, 1),
    "Apple___Cedar_apple_rust": (0, 6),
    "Apple___healthy": (0, 0),
    
    # Corn (1)
    "Corn_(maize)___Common_rust_": (1, 7),
    "Corn_(maize)___Northern_Leaf_Blight": (1, 8),
    "Corn_(maize)___healthy": (1, 0),
    
    # Grape (2)
    "Grape___Black_rot": (2, 1),
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": (2, 8),
    "Grape___healthy": (2, 0),
    
    # Potato (3)
    "Potato___Early_blight": (3, 2),
    "Potato___Late_blight": (3, 3),
    "Potato___healthy": (3, 0),
    
    # Tomato (4)
    "Tomato___Bacterial_spot": (4, 4),
    "Tomato___Early_blight": (4, 2),
    "Tomato___Late_blight": (4, 3),
    "Tomato___Target_Spot": (4, 9),
    "Tomato___healthy": (4, 0)
}

class UploadedSubSetDataset(Dataset):
    def __init__(self, root_dir, parser_dict, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []

        for folder_name, (species_idx, disease_idx) in parser_dict.items():
            folder_path = os.path.join(root_dir, folder_name)
            if os.path.exists(folder_path):
                for file_name in os.listdir(folder_path):
                    if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        img_path = os.path.join(folder_path, file_name)
                        self.samples.append((img_path, species_idx, disease_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, species_target, disease_target = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return (
            image, 
            torch.tensor(species_target, dtype=torch.long), 
            torch.tensor(disease_target, dtype=torch.long)
        )

# 1. Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.12), value=0)
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Instantiate Base Dataset without transforms to split indices cleanly
full_dataset_raw = UploadedSubSetDataset(DATASET_PATH, SUBSET_FOLDER_PARSER, transform=None)
total_samples = len(full_dataset_raw)
indices = list(range(total_samples))

# Extract labels for stratified splitting (balances disease/species distributions across splits)
labels_for_stratify = [f"{sp}_{ds}" for _, sp, ds in full_dataset_raw.samples]

# Split into 70% Train, 15% Val, 15% Test
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, random_state=42, stratify=labels_for_stratify
)
temp_labels = [labels_for_stratify[i] for i in temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=42, stratify=temp_labels
)

# 3. Separate Dataset Instances to isolate Transforms
train_dataset = Subset(UploadedSubSetDataset(DATASET_PATH, SUBSET_FOLDER_PARSER, transform=train_transforms), train_idx)
val_dataset   = Subset(UploadedSubSetDataset(DATASET_PATH, SUBSET_FOLDER_PARSER, transform=eval_transforms), val_idx)
test_dataset  = Subset(UploadedSubSetDataset(DATASET_PATH, SUBSET_FOLDER_PARSER, transform=eval_transforms), test_idx)

# 4. DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Dataset Split Complete! Total: {total_samples}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Dataset Split Complete! Total: 19372
Train: 13560 | Val: 2906 | Test: 2906


In [3]:
import time
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define class lists for beautiful confusion matrix labeling
species_classes = ["Apple", "Corn", "Grape", "Potato", "Tomato"]
disease_classes = ["Healthy", "Black rot", "Early blight", "Late blight", "Bacterial spot", 
                   "Apple scab", "Cedar apple rust", "Common rust", "Northern Leaf Blight", "Target Spot"]

# ==========================================
# 1. HELPER FUNCTION TO INITIALISE BASE CNNs
# ==========================================
def get_model(model_name, num_classes):
    """Loads a vanilla model and changes its final classification layer."""
    if model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        # EfficientNet classifier is Sequential(Dropout, Linear) -> index 1 is the Linear layer
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        
    elif model_name == "mobilenet_v3":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        # MobileNetV3 classifier is Sequential(Linear, Hardswish, Dropout, Linear) -> index 3 or index 1 depending on version
        # To be completely safe across all PyTorch versions, we look at the last element
        if hasattr(model.classifier, 'in_features'):
            in_features = model.classifier.in_features
            model.classifier = nn.Linear(in_features, num_classes)
        else:
            in_features = model.classifier[-1].in_features
            model.classifier[-1] = nn.Linear(in_features, num_classes)
        
    elif model_name == "shufflenet_v2":
        model = models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.DEFAULT)
        # ShuffleNetV2 has a single fc layer out of the box
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    return model

# ==========================================
# 2. STRIP-DOWN PUBLICATION PLOTTING FUNCTION
# ==========================================
def save_publication_cm(cm, class_names, model_name, task_name):
    """Saves a high-res, clean 300 DPI confusion matrix image to disk."""
    plt.figure(figsize=(10, 8), dpi=300)
    sns.set_theme(style="white")
    
    # Using academic 'Blues' color profile
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 9, "weight": "bold"})
    
    plt.title(f"Confusion Matrix: {model_name.upper()} ({task_name.upper()})", fontsize=14, pad=15, weight='bold')
    plt.ylabel('Actual Label', fontsize=12, labelpad=10)
    plt.xlabel('Predicted Label', fontsize=12, labelpad=10)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    # Save file directly into Kaggle output workspace
    filename = f"{model_name}_{task_name}_confusion_matrix.png"
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close() # Closes the plot window to free memory during long execution loops
    print(f" Saved: {filename}")

# ==========================================
# 3. CORE TRAINING & EVALUATION ENGINE
# ==========================================
def train_and_evaluate(model_name, task_name, num_classes, class_names, epochs=10):
    print(f"\n=========================================")
    print(f" TRAINING: {model_name.upper()} | TASK: {task_name.upper()}")
    print(f"=========================================")
    
    model = get_model(model_name, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    target_index = 1 if task_name == "species" else 2
    
    # --- Training Loop ---
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        start_time = time.time()
        
        for images, species_labels, disease_labels in train_loader:
            images = images.to(device)
            targets = species_labels.to(device) if target_index == 1 else disease_labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Time: {elapsed:.1f}s")
        
    # --- Validation Loop ---
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, species_labels, disease_labels in val_loader:
            images = images.to(device)
            targets = species_labels if target_index == 1 else disease_labels
            
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())
            
    # --- Performance Scoring ---
    acc = accuracy_score(all_targets, all_preds)
    prec = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_targets, all_preds)
    
    # Automate saving image for the paper's diagnostic review
    save_publication_cm(cm, class_names, model_name, task_name)
    
    return {
        "Model": model_name,
        "Task": task_name,
        "Accuracy (%)": round(acc * 100, 2),
        "Macro Precision (%)": round(prec * 100, 2),
        "Macro Recall (%)": round(rec * 100, 2),
        "Macro F1-Score (%)": round(f1 * 100, 2)
    }

# ==========================================
# 4. OVERNIGHT EXECUTION PROTOCOL
# ==========================================
tasks = [
    {"name": "species", "classes": 5, "names": species_classes},
    {"name": "disease", "classes": 10, "names": disease_classes}
]
candidate_models = ["efficientnet_b0", "mobilenet_v3", "shufflenet_v2"]

TOTAL_EPOCHS = 10 
all_results = []

for task in tasks:
    for model_name in candidate_models:
        metrics_dict = train_and_evaluate(
            model_name=model_name, 
            task_name=task["name"], 
            num_classes=task["classes"], 
            class_names=task["names"],
            epochs=TOTAL_EPOCHS
        )
        all_results.append(metrics_dict)

# ==========================================
# 5. OUTPUT PRINT MATRIX
# ==========================================
results_df = pd.DataFrame(all_results)
print("\n========================================================")
print("           OVERNIGHT SCREENING MATRIX COMPLETE           ")
print("========================================================")
print(results_df.to_string(index=False))

# Optional: Save text table to csv just in case notebook session disconnects
results_df.to_csv("screening_ablation_results.csv", index=False)

Using device: cuda

 TRAINING: EFFICIENTNET_B0 | TASK: SPECIES
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 123MB/s] 


Epoch [1/10] | Loss: 0.1218 | Time: 214.8s
Epoch [2/10] | Loss: 0.0577 | Time: 85.8s
Epoch [3/10] | Loss: 0.0385 | Time: 83.4s
Epoch [4/10] | Loss: 0.0331 | Time: 92.5s
Epoch [5/10] | Loss: 0.0254 | Time: 95.4s
Epoch [6/10] | Loss: 0.0392 | Time: 85.6s
Epoch [7/10] | Loss: 0.0226 | Time: 85.4s
Epoch [8/10] | Loss: 0.0197 | Time: 82.7s
Epoch [9/10] | Loss: 0.0281 | Time: 83.1s
Epoch [10/10] | Loss: 0.0207 | Time: 80.7s
 Saved: efficientnet_b0_species_confusion_matrix.png

 TRAINING: MOBILENET_V3 | TASK: SPECIES
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 131MB/s] 


Epoch [1/10] | Loss: 0.1095 | Time: 75.9s
Epoch [2/10] | Loss: 0.0510 | Time: 74.9s
Epoch [3/10] | Loss: 0.0548 | Time: 75.3s
Epoch [4/10] | Loss: 0.0336 | Time: 75.6s
Epoch [5/10] | Loss: 0.0269 | Time: 71.4s
Epoch [6/10] | Loss: 0.0288 | Time: 75.0s
Epoch [7/10] | Loss: 0.0174 | Time: 75.3s
Epoch [8/10] | Loss: 0.0898 | Time: 77.1s
Epoch [9/10] | Loss: 0.0279 | Time: 76.6s
Epoch [10/10] | Loss: 0.0207 | Time: 76.1s
 Saved: mobilenet_v3_species_confusion_matrix.png

 TRAINING: SHUFFLENET_V2 | TASK: SPECIES
Downloading: "https://download.pytorch.org/models/shufflenetv2_x1-5666bf0f80.pth" to /root/.cache/torch/hub/checkpoints/shufflenetv2_x1-5666bf0f80.pth


100%|██████████| 8.79M/8.79M [00:00<00:00, 83.3MB/s]


Epoch [1/10] | Loss: 0.1933 | Time: 74.0s
Epoch [2/10] | Loss: 0.0557 | Time: 72.7s
Epoch [3/10] | Loss: 0.0453 | Time: 73.4s
Epoch [4/10] | Loss: 0.0369 | Time: 72.7s
Epoch [5/10] | Loss: 0.0323 | Time: 73.8s
Epoch [6/10] | Loss: 0.0311 | Time: 71.8s
Epoch [7/10] | Loss: 0.0216 | Time: 72.7s
Epoch [8/10] | Loss: 0.0266 | Time: 72.2s
Epoch [9/10] | Loss: 0.0240 | Time: 70.5s
Epoch [10/10] | Loss: 0.0176 | Time: 71.2s
 Saved: shufflenet_v2_species_confusion_matrix.png

 TRAINING: EFFICIENTNET_B0 | TASK: DISEASE
Epoch [1/10] | Loss: 0.3072 | Time: 80.9s
Epoch [2/10] | Loss: 0.1348 | Time: 80.5s
Epoch [3/10] | Loss: 0.1063 | Time: 79.6s
Epoch [4/10] | Loss: 0.0935 | Time: 79.4s
Epoch [5/10] | Loss: 0.0754 | Time: 79.9s
Epoch [6/10] | Loss: 0.0757 | Time: 80.3s
Epoch [7/10] | Loss: 0.0679 | Time: 79.8s
Epoch [8/10] | Loss: 0.0660 | Time: 80.0s
Epoch [9/10] | Loss: 0.0605 | Time: 81.0s
Epoch [10/10] | Loss: 0.0563 | Time: 79.8s
 Saved: efficientnet_b0_disease_confusion_matrix.png

 TRAINING

In [4]:
# ==========================================================
# CELL 3 - DualTaskPlantNet (ShuffleNetV2 + PA + CA + Uncertainty Loss)
# Run AFTER the data cell (needs: train_loader, val_loader, test_loader,
# SUBSET_FOLDER_PARSER).
# ==========================================================
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
 
species_classes = ["Apple", "Corn", "Grape", "Potato", "Tomato"]
disease_classes = ["Healthy", "Black rot", "Early blight", "Late blight", "Bacterial spot",
                   "Apple scab", "Cedar apple rust", "Common rust", "Northern Leaf Blight", "Target Spot"]
 
# Valid (species, disease) pairs, taken straight from the folder parser.
# Used for the "species-disease inconsistency rate" metric (evidence for CA).
VALID_PAIRS = set(SUBSET_FOLDER_PARSER.values())
 
 
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
 

Using device: cuda


In [5]:
# ==========================================================
# 1. ATTENTION MODULES (static shapes, 1x1 convs only -> TensorRT friendly)
# ==========================================================
class PyramidAttention(nn.Module):
    """
    Multi-scale spatial attention from three static pooling levels:
      global (1x1)  : image-level context
      coarse (~3x3) : regional context
      fine   (HxW)  : local 3x3 average, keeps small lesions/spots
    Each level -> 1x1 conv -> resized to HxW -> concat -> 1x1 conv -> sigmoid.
    Output is a residual re-weighting: x + x * attention.
    No BatchNorm on the pooled descriptors, so batch size 1 never breaks training.
    """
    def __init__(self, channels, feat_size=7, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.feat_size = feat_size
 
        coarse_stride = max(feat_size // 3, 1)
        coarse_kernel = feat_size - 2 * coarse_stride      # 7 -> k=3,s=2 -> 3x3 output
        self.pool_global = nn.AvgPool2d(kernel_size=feat_size)
        self.pool_coarse = nn.AvgPool2d(kernel_size=coarse_kernel, stride=coarse_stride)
        self.pool_fine = nn.AvgPool2d(kernel_size=3, stride=1, padding=1, count_include_pad=False)
 
        def reducer():
            return nn.Sequential(nn.Conv2d(channels, mid, 1), nn.ReLU(inplace=True))
 
        self.red_global = reducer()
        self.red_coarse = reducer()
        self.red_fine = reducer()
        self.fuse = nn.Sequential(nn.Conv2d(3 * mid, channels, 1), nn.Sigmoid())
 
    def forward(self, x):
        size = (self.feat_size, self.feat_size)             # static size, no dynamic shapes
        g = F.interpolate(self.red_global(self.pool_global(x)), size=size, mode="nearest")
        c = F.interpolate(self.red_coarse(self.pool_coarse(x)), size=size,
                          mode="bilinear", align_corners=False)
        f = self.red_fine(self.pool_fine(x))
        att = self.fuse(torch.cat([g, c, f], dim=1))
        return x + x * att
 
 
class CrossBranchAttention(nn.Module):
    """
    Channel-wise exchange between Species and Disease branches.
    Both branches' global descriptors are concatenated, and each branch gets its own
    sigmoid channel gate computed from that joint descriptor. The species geometry can
    therefore switch off disease channels that are impossible for the observed species
    (and disease evidence can refine the species features in return).
    """
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
 
        def gate():
            g = nn.Sequential(
                nn.Conv2d(2 * channels, mid, 1), nn.ReLU(inplace=True),
                nn.Conv2d(mid, channels, 1), nn.Sigmoid())
            nn.init.constant_(g[2].bias, 2.0)   # start near identity (sigmoid(2)~0.88)
            return g
 
        self.gate_species = gate()
        self.gate_disease = gate()
 
    def forward(self, f_species, f_disease):
        joint = torch.cat([self.pool(f_species), self.pool(f_disease)], dim=1)   # (B, 2C, 1, 1)
        return f_species * self.gate_species(joint), f_disease * self.gate_disease(joint)
 

In [6]:
# ==========================================================
# 2. DualTaskPlantNet
# ==========================================================
class DualTaskPlantNet(nn.Module):
    def __init__(self, num_species=5, num_disease=10,
                 use_pa=False, use_ca=False, use_uncertainty=False,
                 branch_channels=256, feat_size=7, dropout=0.2):
        super().__init__()
        base = models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.DEFAULT)
 
        # Shared feature extractor: (B,3,224,224) -> (B,1024,7,7)
        self.stem = nn.Sequential(base.conv1, base.maxpool)
        self.stages = nn.Sequential(base.stage2, base.stage3, base.stage4, base.conv5)
 
        def branch_block():
            # 1x1 reduce + depthwise 3x3 keeps each branch specialised but cheap
            return nn.Sequential(
                nn.Conv2d(1024, branch_channels, 1, bias=False),
                nn.BatchNorm2d(branch_channels), nn.ReLU(inplace=True),
                nn.Conv2d(branch_channels, branch_channels, 3, padding=1,
                          groups=branch_channels, bias=False),
                nn.BatchNorm2d(branch_channels), nn.ReLU(inplace=True))
 
        self.species_branch = branch_block()
        self.disease_branch = branch_block()
 
        self.pa_species = PyramidAttention(branch_channels, feat_size) if use_pa else nn.Identity()
        self.pa_disease = PyramidAttention(branch_channels, feat_size) if use_pa else nn.Identity()
        self.ca = CrossBranchAttention(branch_channels) if use_ca else None
 
        self.pool = nn.AdaptiveAvgPool2d(1)
        # Heads are 1x1 convs (not nn.Linear) for TensorRT
        self.species_head = nn.Sequential(nn.Dropout(dropout), nn.Conv2d(branch_channels, num_species, 1))
        self.disease_head = nn.Sequential(nn.Dropout(dropout), nn.Conv2d(branch_channels, num_disease, 1))
 
        # Learnable log-variances for uncertainty weighting: [species, disease]
        if use_uncertainty:
            self.log_vars = nn.Parameter(torch.zeros(2))
        else:
            self.register_parameter("log_vars", None)
 
    def forward(self, x):
        feat = self.stages(self.stem(x))
        fs = self.pa_species(self.species_branch(feat))
        fd = self.pa_disease(self.disease_branch(feat))
        if self.ca is not None:
            fs, fd = self.ca(fs, fd)
        species_logits = self.species_head(self.pool(fs)).flatten(1)
        disease_logits = self.disease_head(self.pool(fd)).flatten(1)
        return species_logits, disease_logits
 

In [7]:
# ==========================================================
# 3. LOSS
#   L = exp(-lv1)*L_sp + exp(-lv2)*L_ds + lv1 + lv2      (Ablation 4)
#   L = L_sp + L_ds                                       (Ablations 1-3)
# ==========================================================
ce_loss = nn.CrossEntropyLoss()
 
 
def compute_loss(model, species_logits, disease_logits, y_species, y_disease):
    ls = ce_loss(species_logits.float(), y_species)
    ld = ce_loss(disease_logits.float(), y_disease)
    if model.log_vars is not None:
        lv = model.log_vars
        total = torch.exp(-lv[0]) * ls + torch.exp(-lv[1]) * ld + lv[0] + lv[1]
    else:
        total = ls + ld
    return total, ls, ld

In [8]:
# ==========================================================
# 4. EVALUATION
# ==========================================================
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    sp_p, ds_p, sp_t, ds_t = [], [], [], []
    for images, y_sp, y_ds in loader:
        images = images.to(device, non_blocking=True)
        s_out, d_out = model(images)
        sp_p.extend(s_out.argmax(1).cpu().numpy())
        ds_p.extend(d_out.argmax(1).cpu().numpy())
        sp_t.extend(y_sp.numpy())
        ds_t.extend(y_ds.numpy())
    sp_p, ds_p, sp_t, ds_t = map(np.array, (sp_p, ds_p, sp_t, ds_t))
 
    def task_metrics(t, p):
        return {
            "acc": accuracy_score(t, p) * 100,
            "prec": precision_score(t, p, average="macro", zero_division=0) * 100,
            "rec": recall_score(t, p, average="macro", zero_division=0) * 100,
            "f1": f1_score(t, p, average="macro", zero_division=0) * 100,
        }
 
    m = {
        "species": task_metrics(sp_t, sp_p),
        "disease": task_metrics(ds_t, ds_p),
        "joint_acc": np.mean((sp_p == sp_t) & (ds_p == ds_t)) * 100,
        # % of predictions whose (species, disease) combo cannot exist in the dataset
        "inconsistency": np.mean([(s, d) not in VALID_PAIRS for s, d in zip(sp_p, ds_p)]) * 100,
        "cm_species": confusion_matrix(sp_t, sp_p, labels=range(len(species_classes))),
        "cm_disease": confusion_matrix(ds_t, ds_p, labels=range(len(disease_classes))),
    }
    m["select_score"] = (m["species"]["f1"] + m["disease"]["f1"]) / 2   # used for best-epoch pick
    return m
 
 
def save_publication_cm(cm, class_names, run_name, task_name):
    plt.figure(figsize=(10, 8), dpi=300)
    sns.set_theme(style="white")
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=True,
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 9, "weight": "bold"})
    plt.title(f"Confusion Matrix: {run_name} ({task_name.upper()})", fontsize=14, pad=15, weight="bold")
    plt.ylabel("Actual Label", fontsize=12, labelpad=10)
    plt.xlabel("Predicted Label", fontsize=12, labelpad=10)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    filename = f"{run_name}_{task_name}_confusion_matrix.png"
    plt.savefig(filename, bbox_inches="tight", dpi=300)
    plt.close()
    print(f" Saved: {filename}")
 
 
@torch.no_grad()
def benchmark_latency(model, runs=100, warmup=20):
    """Batch-1 latency (ms). Relative comparison only - Kaggle GPU != Jetson Nano."""
    model.eval()
    x = torch.randn(1, 3, 224, 224, device=device)
    for _ in range(warmup):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(runs):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / runs * 1000
 

In [9]:
# ==========================================================
# 5. TRAINING ONE ABLATION
#    - trains on train_loader
#    - picks best epoch on val_loader
#    - reports final numbers on the held-out test_loader
# ==========================================================
def run_ablation(name, use_pa, use_ca, use_uncertainty, epochs=25, lr=1e-3, seed=42):
    print(f"\n{'=' * 60}\n ABLATION: {name} | PA={use_pa} CA={use_ca} UncertaintyLoss={use_uncertainty}\n{'=' * 60}")
    set_seed(seed)
 
    model = DualTaskPlantNet(use_pa=use_pa, use_ca=use_ca, use_uncertainty=use_uncertainty).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
 
    # No weight decay on the log-variance parameters
    decay = [p for n, p in model.named_parameters() if n != "log_vars"]
    groups = [{"params": decay, "weight_decay": 1e-4}]
    if model.log_vars is not None:
        groups.append({"params": [model.log_vars], "weight_decay": 0.0})
    optimizer = torch.optim.AdamW(groups, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
 
    best_score, best_epoch = -1.0, 0
    ckpt_path = f"{name}_best.pth"
 
    for epoch in range(epochs):
        model.train()
        t0 = time.time()
        run_loss = 0.0
        for images, y_sp, y_ds in train_loader:
            images = images.to(device, non_blocking=True)
            y_sp, y_ds = y_sp.to(device), y_ds.to(device)
 
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                s_out, d_out = model(images)
            loss, ls, ld = compute_loss(model, s_out, d_out, y_sp, y_ds)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item() * images.size(0)
        scheduler.step()
 
        val = evaluate(model, val_loader)
        msg = (f"Epoch [{epoch + 1:02d}/{epochs}] loss {run_loss / len(train_loader.dataset):.4f} | "
               f"val F1 sp {val['species']['f1']:.2f} ds {val['disease']['f1']:.2f} | "
               f"inconsist {val['inconsistency']:.2f}% | {time.time() - t0:.0f}s")
        if model.log_vars is not None:
            w = torch.exp(-model.log_vars.detach()).cpu().numpy()
            msg += f" | task weights sp {w[0]:.2f} ds {w[1]:.2f}"
        print(msg)
 
        if val["select_score"] > best_score:
            best_score, best_epoch = val["select_score"], epoch + 1
            torch.save(model.state_dict(), ckpt_path)
 
    # ---- Final test evaluation with the best-val checkpoint ----
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    test = evaluate(model, test_loader)
    latency = benchmark_latency(model)
    print(f" Best epoch: {best_epoch} | TEST species F1 {test['species']['f1']:.2f} | "
          f"disease F1 {test['disease']['f1']:.2f} | inconsistency {test['inconsistency']:.2f}%")
 
    save_publication_cm(test["cm_species"], species_classes, name, "species")
    save_publication_cm(test["cm_disease"], disease_classes, name, "disease")
 
    row = {
        "Ablation": name,
        "PA": use_pa, "CA": use_ca, "Uncertainty Loss": use_uncertainty,
        "Best Epoch": best_epoch,
        "Species Acc (%)": round(test["species"]["acc"], 2),
        "Species Macro F1 (%)": round(test["species"]["f1"], 2),
        "Disease Acc (%)": round(test["disease"]["acc"], 2),
        "Disease Macro P (%)": round(test["disease"]["prec"], 2),
        "Disease Macro R (%)": round(test["disease"]["rec"], 2),
        "Disease Macro F1 (%)": round(test["disease"]["f1"], 2),
        "Joint Acc (%)": round(test["joint_acc"], 2),
        "Species-Disease Inconsistency (%)": round(test["inconsistency"], 2),
        "Params (M)": round(n_params, 3),
        "Latency bs1 (ms)": round(latency, 2),
    }
    if model.log_vars is not None:
        row["log_var_species"] = round(model.log_vars[0].item(), 4)
        row["log_var_disease"] = round(model.log_vars[1].item(), 4)
    return row
 
 

In [10]:
# ==========================================================
# 6. ABLATION PLAN  (Table 1)
#    Ablations 1-3 use fixed equal loss weights (L_sp + L_ds);
#    only Ablation 4 uses uncertainty weighting.
# ==========================================================
ABLATIONS = [
    dict(name="A1_base_dualtask",   use_pa=False, use_ca=False, use_uncertainty=False),
    dict(name="A2_cross_attn_only", use_pa=False, use_ca=True,  use_uncertainty=False),
    dict(name="A3_pyramid_only",    use_pa=True,  use_ca=False, use_uncertainty=False),
    dict(name="A4_proposed_full",   use_pa=True,  use_ca=True,  use_uncertainty=True),
]
 
EPOCHS = 25
ABLATIONS_TO_RUN = [0, 1, 2, 3]     # e.g. [0, 1] in one Kaggle session, [2, 3] in the next
 
all_results = []
for i in ABLATIONS_TO_RUN:
    row = run_ablation(epochs=EPOCHS, **ABLATIONS[i])
    all_results.append(row)
    # Save after every run so a disconnected session doesn't lose finished ablations
    pd.DataFrame(all_results).to_csv("ablation_results.csv", index=False)
 
results_df = pd.DataFrame(all_results)
print("\n" + "=" * 60 + "\n ABLATION TABLE (TEST SET)\n" + "=" * 60)
print(results_df.to_string(index=False))
 


 ABLATION: A1_base_dualtask | PA=False CA=False UncertaintyLoss=False
Epoch [01/25] loss 0.4651 | val F1 sp 99.13 ds 95.89 | inconsist 0.10% | 91s
Epoch [02/25] loss 0.1860 | val F1 sp 99.15 ds 97.60 | inconsist 0.07% | 79s
Epoch [03/25] loss 0.1404 | val F1 sp 99.30 ds 97.88 | inconsist 0.21% | 79s
Epoch [04/25] loss 0.1276 | val F1 sp 99.64 ds 98.03 | inconsist 0.24% | 78s
Epoch [05/25] loss 0.0950 | val F1 sp 99.48 ds 98.05 | inconsist 0.07% | 78s
Epoch [06/25] loss 0.0918 | val F1 sp 99.42 ds 98.74 | inconsist 0.17% | 79s
Epoch [07/25] loss 0.0869 | val F1 sp 99.72 ds 98.86 | inconsist 0.03% | 80s
Epoch [08/25] loss 0.0743 | val F1 sp 99.43 ds 98.96 | inconsist 0.24% | 80s
Epoch [09/25] loss 0.0663 | val F1 sp 99.68 ds 98.15 | inconsist 0.14% | 81s
Epoch [10/25] loss 0.0523 | val F1 sp 99.81 ds 98.71 | inconsist 0.07% | 83s
Epoch [11/25] loss 0.0342 | val F1 sp 99.84 ds 98.80 | inconsist 0.03% | 77s
Epoch [12/25] loss 0.0432 | val F1 sp 99.74 ds 99.13 | inconsist 0.03% | 80s
Epoch

In [11]:
# ==========================================================
# CELL 4 - Pick the best ablation, print it, save final .pth
# Run AFTER the ablation cell (needs: all_results / results_df,
# and the "{name}_best.pth" checkpoint files it saved).
# ==========================================================
import shutil
import pandas as pd

results_df = pd.DataFrame(all_results)  # rebuild in case this cell runs standalone after a reload
# results_df = pd.read_csv("ablation_results.csv")   # <- use this instead if resuming in a new session

# Combined score = average of the two task Macro F1s (same criterion used for best-epoch selection)
results_df["Combined Macro F1 (%)"] = (
    results_df["Species Macro F1 (%)"] + results_df["Disease Macro F1 (%)"]
) / 2

best_row = results_df.loc[results_df["Combined Macro F1 (%)"].idxmax()]
best_name = best_row["Ablation"]

print("=" * 60)
print(" BEST ABLATION (by combined Macro F1 across both tasks)")
print("=" * 60)
print(best_row.to_string())

# Copy that ablation's checkpoint to a clearly-named final file in Kaggle's working dir
src_ckpt = f"{best_name}_best.pth"
final_ckpt = "/kaggle/working/best_model_final.pth"
shutil.copy(src_ckpt, final_ckpt)
print(f"\nSaved best model checkpoint -> {final_ckpt}")

# Also save the full results table alongside it for the paper's Table 1
results_df.to_csv("/kaggle/working/ablation_results_final.csv", index=False)
print("Saved full ablation table -> /kaggle/working/ablation_results_final.csv")

 BEST ABLATION (by combined Macro F1 across both tasks)
Ablation                             A3_pyramid_only
PA                                              True
CA                                             False
Uncertainty Loss                               False
Best Epoch                                        24
Species Acc (%)                                99.86
Species Macro F1 (%)                           99.85
Disease Acc (%)                                99.86
Disease Macro P (%)                            99.87
Disease Macro R (%)                            99.88
Disease Macro F1 (%)                           99.88
Joint Acc (%)                                  99.76
Species-Disease Inconsistency (%)                0.0
Params (M)                                     1.986
Latency bs1 (ms)                                8.69
log_var_species                                  NaN
log_var_disease                                  NaN
Combined Macro F1 (%)                      